# Sistema de control presupuestario y KPIs financieros
Proyecto final - Data Analyst + IA

Fases: (1) exploracion SQL, (2) limpieza y logica de negocio en Python, (3) consumo de API externa via GET, (4) calculo de KPIs para Power BI.

## 1. Conexion a la base de datos SQL

In [2]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect('../data/sql/control_presupuestario.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,departamentos
1,categorias
2,periodos
3,presupuesto
4,ejecucion_real_raw
5,ingresos


## 2. Limpieza de `ejecucion_real_raw` con pandas
Aplicar aqui la misma logica de integridad de datos ya trabajada en el curso: normalizar nombres de departamento, convertir montos a numerico, marcar (flag) valores nulos/negativos/duplicados antes de usarlos en el analisis.

In [2]:
df_raw = pd.read_sql('SELECT * FROM ejecucion_real_raw', conn)
df_raw.head()

,id_ejecucion,departamento,categoria,anio,mes,monto_ejecutado,fecha_registro
0,1,Atencion al Cliente,Nomina y Personal,2025,1,42692.08,2025-01-09
1,2,At. Cliente,Nomina y Personal,2025,2,43456.48,2025-02-21
2,3,Atencion al Cliente,Nomina y Personal,2025,3,"42,426.84 EUR",2025-03-15
3,4,ATENCION AL CLIENTE,Nomina y Personal,2025,4,43062.15,2025-04-05
4,5,Atencion al Cliente,Nomina y Personal,2025,5,43880.34,2025-05-06


### 2.1 Normalizar el nombre de departamento
El sistema origen registra el mismo departamento con distintas variantes (mayusculas, abreviaturas, espacios). Se mapea cada variante detectada a su nombre canonico antes de cruzar con la tabla `departamentos`.

In [3]:
mapa_departamentos = {
    'atencion al cliente': 'Atencion al Cliente', 'atencion cliente': 'Atencion al Cliente',
    'at. cliente': 'Atencion al Cliente',
    'ventas b2b': 'Ventas B2B', 'ventas corporativas b2b': 'Ventas B2B',
    'operaciones y logistica': 'Operaciones y Logistica', 'operaciones/logistica': 'Operaciones y Logistica',
    'marketing': 'Marketing', 'mercadeo': 'Marketing',
    'tecnologia y ti': 'Tecnologia y TI', 'ti': 'Tecnologia y TI', 'tecnologia/ti': 'Tecnologia y TI',
    'administracion y finanzas': 'Administracion y Finanzas', 'admin y finanzas': 'Administracion y Finanzas',
}

def normalizar_departamento(nombre):
    return mapa_departamentos.get(str(nombre).strip().lower(), str(nombre).strip())

df_raw['departamento_normalizado'] = df_raw['departamento'].apply(normalizar_departamento)

# Validacion: no deberia quedar ningun nombre sin mapear a la dimension
df_departamentos = pd.read_sql('SELECT * FROM departamentos', conn)
no_mapeados = set(df_raw['departamento_normalizado']) - set(df_departamentos['nombre'])
assert not no_mapeados, f'Quedaron sin mapear: {no_mapeados}'
print('Todos los departamentos se mapearon correctamente.')

Todos los departamentos se mapearon correctamente.


### 2.2 Convertir `monto_ejecutado` a numerico
El campo llega como texto con varios formatos invalidos: vacio, `'N/D'`, con simbolo de moneda (`'4,894.70 EUR'`), o en formato decimal europeo (`'3880,15'`). Se normaliza todo a `float`, devolviendo `NaN` cuando no es reconstruible.

In [4]:
def limpiar_monto(valor):
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)
    texto = str(valor).strip().replace('EUR', '').replace('€', '').strip()
    if texto == '' or texto.upper() == 'N/D':
        return np.nan
    if ',' in texto and '.' in texto:
        texto = texto.replace(',', '')      # separador de miles: 4,894.70
    elif ',' in texto and '.' not in texto:
        texto = texto.replace(',', '.')     # formato europeo: 3880,15
    try:
        return float(texto)
    except ValueError:
        return np.nan

df_raw['monto_numerico'] = df_raw['monto_ejecutado'].apply(limpiar_monto)
print('Valores no reconstruibles (nulos tras conversion):', df_raw['monto_numerico'].isna().sum())

Valores no reconstruibles (nulos tras conversion): 10


### 2.3 Flags de integridad de datos
En vez de borrar o corregir en silencio, se marca (flag) cada anomalia para dejar trazabilidad de las decisiones tomadas:
- **Nulos**: no se pueden reconstruir -> se excluyen del dataset limpio, pero quedan documentados.
- **Negativos**: son error de captura de signo (ningun gasto es negativo en este contexto) -> se corrige con valor absoluto, y se deja el flag como evidencia de la correccion.
- **Duplicados**: mismo departamento/categoria/periodo/monto/fecha -> se conserva solo la primera aparicion.

In [5]:
df_raw['flag_valor_nulo'] = df_raw['monto_numerico'].isna()
df_raw['flag_valor_negativo'] = df_raw['monto_numerico'] < 0
df_raw['monto_corregido'] = df_raw['monto_numerico'].abs()

cols_dup = ['departamento_normalizado', 'categoria', 'anio', 'mes', 'monto_numerico', 'fecha_registro']
df_raw['flag_duplicado'] = df_raw.duplicated(subset=cols_dup, keep='first')

total = len(df_raw)
print('--- Resumen de calidad de datos ---')
print(f"Total filas raw: {total}")
print(f"Valores nulos/no numericos: {df_raw['flag_valor_nulo'].sum()} ({df_raw['flag_valor_nulo'].sum()/total*100:.1f}%)")
print(f"Valores negativos corregidos: {df_raw['flag_valor_negativo'].sum()} ({df_raw['flag_valor_negativo'].sum()/total*100:.1f}%)")
print(f"Filas duplicadas eliminadas: {df_raw['flag_duplicado'].sum()} ({df_raw['flag_duplicado'].sum()/total*100:.1f}%)")

--- Resumen de calidad de datos ---
Total filas raw: 241
Valores nulos/no numericos: 10 (4.1%)
Valores negativos corregidos: 5 (2.1%)
Filas duplicadas eliminadas: 1 (0.4%)


### 2.4 Construir el dataset limpio (listo para el esquema estrella)
Se excluyen nulos y duplicados, se usa el monto corregido, y se traducen los nombres normalizados a sus IDs de dimension (`departamento_id`, `categoria_id`, `periodo_id`) para que la tabla quede con la misma forma que `presupuesto` e `ingresos`.

In [6]:
df_categorias = pd.read_sql('SELECT * FROM categorias', conn)
df_periodos = pd.read_sql('SELECT * FROM periodos', conn)

df_limpio = df_raw[~df_raw['flag_valor_nulo'] & ~df_raw['flag_duplicado']].copy()

df_limpio = df_limpio.merge(df_departamentos, left_on='departamento_normalizado', right_on='nombre')
df_limpio = df_limpio.merge(df_categorias, left_on='categoria', right_on='nombre', suffixes=('', '_cat'))
df_limpio = df_limpio.merge(df_periodos, on=['anio', 'mes'])

ejecucion_real_limpia = df_limpio[[
    'departamento_id', 'categoria_id', 'periodo_id', 'monto_corregido', 'flag_valor_negativo'
]].rename(columns={'monto_corregido': 'monto_ejecutado', 'flag_valor_negativo': 'fue_corregido_signo'})
ejecucion_real_limpia.insert(0, 'id_ejecucion', range(1, len(ejecucion_real_limpia) + 1))

print(f"Filas en dataset limpio: {len(ejecucion_real_limpia)} de {total} originales")
ejecucion_real_limpia.head()

Filas en dataset limpio: 230 de 241 originales


,id_ejecucion,departamento_id,categoria_id,periodo_id,monto_ejecutado,fue_corregido_signo
0,1,1,1,1,42692.08,False
1,2,1,1,2,43456.48,False
2,3,1,1,3,42426.84,False
3,4,1,1,4,43062.15,False
4,5,1,1,5,43880.34,False


In [7]:
# Guardar resultado limpio: CSV para revision + tabla en SQLite para Power BI
ejecucion_real_limpia.to_csv('../data/processed/ejecucion_real_limpia.csv', index=False)

ejecucion_real_limpia.to_sql('ejecucion_real_limpia', conn, if_exists='replace', index=False)
conn.commit()
print('Guardado en data/processed/ejecucion_real_limpia.csv y en la tabla ejecucion_real_limpia de la base de datos')

Guardado en data/processed/ejecucion_real_limpia.csv y en la tabla ejecucion_real_limpia de la base de datos


## 3. Consumo de API externa (metodo GET)
Tipo de cambio EUR/USD historico (Frankfurter API - BCE), para analizar el impacto cambiario en categorias con gasto en dolares.

In [ ]:
import sys
sys.path.append('../src')
from obtener_tipo_cambio import obtener_tipo_cambio

# Ejemplo de una sola consulta
obtener_tipo_cambio('2025-06-01')

## 4. KPIs financieros
Se calculan a nivel detallado (departamento x categoria x mes) para poder filtrar en Power BI, y tambien se guardan como tabla propia lista para importar directamente.

- **% ejecucion** = ejecutado / presupuestado
- **Desviacion absoluta y porcentual** = ejecutado - presupuestado
- **Ejecucion acumulada (YTD)** = suma acumulada mes a mes por departamento/categoria
- **Flag de alerta** = desviacion > 10% (umbral de sobreejecucion)

Se usa LEFT JOIN desde `presupuesto` hacia `ejecucion_real_limpia` para no perder filas de presupuesto que no tuvieron dato de ejecucion valido (los 10 registros excluidos en la limpieza).

In [8]:
df_pres = pd.read_sql('SELECT * FROM presupuesto', conn)

kpi = df_pres.merge(
    ejecucion_real_limpia[['departamento_id','categoria_id','periodo_id','monto_ejecutado']],
    on=['departamento_id','categoria_id','periodo_id'], how='left'
)
kpi = kpi.merge(df_periodos, on='periodo_id')
kpi = kpi.merge(df_departamentos, on='departamento_id').rename(columns={'nombre': 'departamento'})
kpi = kpi.merge(df_categorias, on='categoria_id').rename(columns={'nombre': 'categoria'})

kpi['sin_dato_ejecucion'] = kpi['monto_ejecutado'].isna()
kpi['pct_ejecucion'] = kpi['monto_ejecutado'] / kpi['monto_presupuestado']
kpi['desviacion_absoluta'] = kpi['monto_ejecutado'] - kpi['monto_presupuestado']
kpi['desviacion_pct'] = kpi['desviacion_absoluta'] / kpi['monto_presupuestado']

UMBRAL_ALERTA = 0.10
kpi['flag_alerta_sobreejecucion'] = kpi['desviacion_pct'] > UMBRAL_ALERTA

# Acumulado YTD por departamento y categoria
kpi = kpi.sort_values(['departamento_id', 'categoria_id', 'anio', 'mes'])
kpi['presupuesto_ytd'] = kpi.groupby(['departamento_id', 'categoria_id'])['monto_presupuestado'].cumsum()
kpi['ejecutado_ytd'] = kpi.groupby(['departamento_id', 'categoria_id'])['monto_ejecutado'].cumsum()
kpi['pct_ejecucion_ytd'] = kpi['ejecutado_ytd'] / kpi['presupuesto_ytd']

print(f"Filas KPI: {len(kpi)} | Sin dato de ejecucion: {kpi['sin_dato_ejecucion'].sum()}")
print(f"Alertas de sobreejecucion (>{UMBRAL_ALERTA:.0%}): {kpi['flag_alerta_sobreejecucion'].sum()}")

kpi[kpi['flag_alerta_sobreejecucion']][['departamento','categoria','mes','pct_ejecucion','desviacion_pct']].round(3).head(10)

Filas KPI: 240 | Sin dato de ejecucion: 10
Alertas de sobreejecucion (>10%): 15


,departamento,categoria,mes,pct_ejecucion,desviacion_pct
42,Marketing,Nomina y Personal,7,1.104,0.104
44,Marketing,Nomina y Personal,9,1.126,0.126
183,Marketing,Gastos Operativos,4,1.128,0.128
188,Marketing,Gastos Operativos,9,1.118,0.118
177,Marketing,Marketing y Publicidad,10,1.117,0.117
50,Tecnologia y TI,Nomina y Personal,3,1.100,0.100
51,Tecnologia y TI,Nomina y Personal,4,1.145,0.145
52,Tecnologia y TI,Nomina y Personal,5,1.112,0.112
55,Tecnologia y TI,Nomina y Personal,8,1.121,0.121
58,Tecnologia y TI,Nomina y Personal,11,1.130,0.130


In [9]:
# Guardar KPI detallado para Power BI
kpi_export = kpi.drop(columns=['departamento', 'categoria', 'anio', 'mes'])
kpi_export.to_csv('../data/processed/kpi_presupuesto_detallado.csv', index=False)
kpi_export.to_sql('kpi_presupuesto_detallado', conn, if_exists='replace', index=False)
conn.commit()
print('Guardado: kpi_presupuesto_detallado (CSV + tabla SQL)')

Guardado: kpi_presupuesto_detallado (CSV + tabla SQL)


## 5. Ingresos y crecimiento
Cargar la tabla `ingresos` (vinculada a departamentos y periodos) y calcular:
- Evolucion mensual de ingresos por linea de negocio
- % de crecimiento interanual (enero vs diciembre)
- Margen operativo = (ingresos - gastos ejecutados) / ingresos

In [10]:
df_ingresos = pd.read_sql('SELECT * FROM ingresos', conn)
df_ingresos.head()

,id_ingreso,departamento_id,concepto,periodo_id,monto_ingresos
0,1,2,Servicios B2B Corporativos,1,258950.55
1,2,2,Servicios B2B Corporativos,2,257646.06
2,3,2,Servicios B2B Corporativos,3,269513.74
3,4,2,Servicios B2B Corporativos,4,269992.46
4,5,2,Servicios B2B Corporativos,5,276210.22


In [11]:
ingresos_mes = (df_ingresos.merge(df_periodos, on='periodo_id')
                .groupby(['periodo_id', 'anio', 'mes'])['monto_ingresos'].sum()
                .reset_index(name='total_ingresos'))

gastos_mes = (ejecucion_real_limpia.merge(df_periodos, on='periodo_id')
              .groupby(['periodo_id', 'anio', 'mes'])['monto_ejecutado'].sum()
              .reset_index(name='total_gastos'))

resumen_mensual = ingresos_mes.merge(gastos_mes, on=['periodo_id', 'anio', 'mes']).sort_values('mes')
resumen_mensual['margen_operativo'] = resumen_mensual['total_ingresos'] - resumen_mensual['total_gastos']
resumen_mensual['margen_operativo_pct'] = resumen_mensual['margen_operativo'] / resumen_mensual['total_ingresos']
resumen_mensual['crecimiento_ingresos_mom'] = resumen_mensual['total_ingresos'].pct_change()

crecimiento_anual = resumen_mensual['total_ingresos'].iloc[-1] / resumen_mensual['total_ingresos'].iloc[0] - 1
print(f"Crecimiento de ingresos enero -> diciembre: {crecimiento_anual*100:.1f}%")
print(f"Margen operativo promedio: {resumen_mensual['margen_operativo_pct'].mean()*100:.1f}%")

resumen_mensual[['mes', 'total_ingresos', 'total_gastos', 'margen_operativo_pct']].round(3)

Crecimiento de ingresos enero -> diciembre: 18.3%
Margen operativo promedio: 29.1%


,mes,total_ingresos,total_gastos,margen_operativo_pct
0,1,475285.00,364072.30,0.234
1,2,479306.77,332116.98,0.307
2,3,494354.03,368224.44,0.255
3,4,495937.39,353320.50,0.288
4,5,510113.12,373165.18,0.268
5,6,505783.25,363579.91,0.281
6,7,495698.39,360503.00,0.273
7,8,466280.51,374899.55,0.196
8,9,534299.61,314041.18,0.412
9,10,535287.52,366279.79,0.316


In [12]:
# Crecimiento por linea de negocio (enero vs diciembre)
por_linea = df_ingresos.merge(df_periodos, on='periodo_id')
tabla_lineas = por_linea.pivot_table(index='mes', columns='concepto', values='monto_ingresos', aggfunc='sum')
crecimiento_lineas = ((tabla_lineas.loc[12] / tabla_lineas.loc[1] - 1) * 100).round(1)
crecimiento_lineas.name = 'crecimiento_pct_ene_dic'
crecimiento_lineas.sort_values(ascending=False)

concepto
Servicios de Valor Añadido (Cloud/Digital)    37.8
Servicios B2B Corporativos                    18.3
Servicios Residenciales                        9.4
Name: crecimiento_pct_ene_dic, dtype: float64

In [13]:
# Guardar resumen mensual para Power BI
resumen_mensual.to_csv('../data/processed/resumen_financiero_mensual.csv', index=False)
resumen_mensual.to_sql('resumen_financiero_mensual', conn, if_exists='replace', index=False)
conn.commit()
print('Guardado: resumen_financiero_mensual (CSV + tabla SQL)')

Guardado: resumen_financiero_mensual (CSV + tabla SQL)


## 6. Forecast 2026
No todo se proyecta igual:
- **Ingresos**: se proyectan estadisticamente (tendencia + estacionalidad), porque son un resultado que depende del mercado.
- **Presupuesto**: es una decision de planificacion (incremento planificado), no una proyeccion estadistica.
- **Gasto esperado**: se estima aplicando el ratio historico de ejecucion (2025) de cada departamento/categoria sobre el presupuesto 2026.

### 6.1 Extender la dimension de periodos a 2026

In [3]:
cur = conn.cursor()
max_id = pd.read_sql('SELECT MAX(periodo_id) as m FROM periodos', conn)['m'][0]
if max_id == 12:  # evitar duplicar si se re-ejecuta
    nuevos_periodos = [(max_id + i, 2026, i) for i in range(1, 13)]
    cur.executemany('INSERT INTO periodos VALUES (?,?,?)', nuevos_periodos)
    conn.commit()
df_periodos = pd.read_sql('SELECT * FROM periodos', conn)
periodo_2026 = {row['mes']: row['periodo_id'] for _, row in df_periodos[df_periodos['anio']==2026].iterrows()}
print(f"Periodos totales: {len(df_periodos)} (12 de 2025 + 12 de 2026)")

Periodos totales: 24 (12 de 2025 + 12 de 2026)


### 6.2 Forecast de ingresos: tendencia + estacionalidad
Con solo 12 meses de historia no alcanza para una descomposicion clasica robusta, asi que se usa una version simplificada y explicable: se ajusta una recta de tendencia por minimos cuadrados (`np.polyfit`), se calcula el indice estacional de cada mes como `real / tendencia`, y se proyecta la tendencia a 2026 reaplicando ese mismo indice estacional mes a mes.

In [15]:
por_linea = (df_ingresos.merge(df_periodos[df_periodos['anio']==2025], on='periodo_id')
             .groupby(['departamento_id','concepto','mes'])['monto_ingresos'].sum().reset_index())

forecast_rows = []
for (dept_id, concepto), grupo in por_linea.groupby(['departamento_id','concepto']):
    grupo = grupo.sort_values('mes')
    x, y = grupo['mes'].values, grupo['monto_ingresos'].values
    pendiente, intercepto = np.polyfit(x, y, 1)
    indice_estacional = y / (pendiente * x + intercepto)
    x_2026 = np.arange(13, 25)
    forecast_vals = (pendiente * x_2026 + intercepto) * indice_estacional
    for mes, monto in zip(range(1, 13), forecast_vals):
        forecast_rows.append([dept_id, concepto, periodo_2026[mes], round(monto, 2)])

ingresos_forecast_2026 = pd.DataFrame(forecast_rows, columns=['departamento_id','concepto','periodo_id','monto_ingresos_forecast'])
ingresos_forecast_2026.insert(0, 'id_forecast', range(1, len(ingresos_forecast_2026)+1))

crecimiento = ingresos_forecast_2026['monto_ingresos_forecast'].sum() / df_ingresos['monto_ingresos'].sum() - 1
print(f"Ingresos 2025: {df_ingresos['monto_ingresos'].sum():,.0f} EUR")
print(f"Forecast 2026: {ingresos_forecast_2026['monto_ingresos_forecast'].sum():,.0f} EUR ({crecimiento*100:+.1f}%)")

Ingresos 2025: 6,095,934 EUR
Forecast 2026: 7,028,269 EUR (+15.3%)


### 6.3 Presupuesto 2026: decision de planificacion
Marketing y Tecnologia (los departamentos que en 2025 invirtieron mas alla del presupuesto para sostener el crecimiento) reciben un incremento planificado del 6%; el resto, un ajuste estandar del 3%.

In [16]:
deptos_crecimiento = df_departamentos[df_departamentos['nombre'].isin(['Marketing','Tecnologia y TI'])]['departamento_id'].tolist()

base_2025 = (df_pres.merge(df_periodos[df_periodos['anio']==2025], on='periodo_id')
             .groupby(['departamento_id','categoria_id'])['monto_presupuestado'].mean().reset_index())

pres_2026_rows = []
for _, row in base_2025.iterrows():
    factor = 1.06 if row['departamento_id'] in deptos_crecimiento else 1.03
    for mes in range(1, 13):
        pres_2026_rows.append([row['departamento_id'], row['categoria_id'], periodo_2026[mes],
                                round(row['monto_presupuestado'] * factor, 2)])

presupuesto_2026 = pd.DataFrame(pres_2026_rows, columns=['departamento_id','categoria_id','periodo_id','monto_presupuestado_2026'])
presupuesto_2026.insert(0, 'id_presupuesto', range(1, len(presupuesto_2026)+1))

crecimiento_pres = presupuesto_2026['monto_presupuestado_2026'].sum() / (base_2025['monto_presupuestado'].sum()*12) - 1
print(f"Presupuesto 2026: {presupuesto_2026['monto_presupuestado_2026'].sum():,.0f} EUR ({crecimiento_pres*100:+.1f}% vs 2025)")

Presupuesto 2026: 4,482,809 EUR (+4.2% vs 2025)


### 6.4 Gasto esperado 2026
Se aplica el ratio historico de `% ejecucion` (2025) de cada departamento/categoria — calculado en la seccion 4 — sobre el nuevo presupuesto 2026. Es una forma de decir: "si este equipo historicamente ejecuta 110% de lo presupuestado, es razonable esperar algo similar el proximo año, salvo que se tomen medidas de control".

In [17]:
ratio_ejec = kpi.groupby(['departamento_id','categoria_id'])['pct_ejecucion'].mean().reset_index()
ratio_ejec.columns = ['departamento_id','categoria_id','ratio']

gasto_esp = presupuesto_2026.merge(ratio_ejec, on=['departamento_id','categoria_id'], how='left')
gasto_esp['ratio'] = gasto_esp['ratio'].fillna(1.0)
gasto_esp['monto_gasto_esperado'] = round(gasto_esp['monto_presupuestado_2026'] * gasto_esp['ratio'], 2)
gasto_esperado_2026 = gasto_esp[['departamento_id','categoria_id','periodo_id','monto_gasto_esperado']].copy()
gasto_esperado_2026.insert(0, 'id_gasto', range(1, len(gasto_esperado_2026)+1))

total_ing_2026 = ingresos_forecast_2026['monto_ingresos_forecast'].sum()
total_gasto_2026 = gasto_esperado_2026['monto_gasto_esperado'].sum()
print(f"Gasto esperado 2026: {total_gasto_2026:,.0f} EUR")
print(f"Margen operativo esperado 2026: {(1 - total_gasto_2026/total_ing_2026)*100:.1f}%")

Gasto esperado 2026: 4,633,122 EUR
Margen operativo esperado 2026: 34.1%


### 6.5 Intervalo de confianza del forecast de ingresos
Un forecast sin rango de incertidumbre es una falsa promesa de precision. Se calcula el error estandar de los residuos de la regresion (real - tendencia) y se construye una banda de confianza del 95% (`± 1.96 x error estandar`) alrededor de cada mes proyectado — practica estandar en proyecciones estadisticas, y lo que distingue una proyeccion rigurosa de una simple extrapolacion.

In [18]:
filas_ic = []
for (dept_id, concepto), grupo in por_linea.groupby(['departamento_id','concepto']):
    grupo = grupo.sort_values('mes')
    x, y = grupo['mes'].values, grupo['monto_ingresos'].values
    pendiente, intercepto = np.polyfit(x, y, 1)
    error_estandar = np.std(y - (pendiente * x + intercepto), ddof=2)
    margen = 1.96 * error_estandar
    filas_ic.append([concepto, error_estandar, margen])

pd.DataFrame(filas_ic, columns=['concepto','error_estandar_mensual','margen_95pct']).round(0)

,concepto,error_estandar_mensual,margen_95pct
0,Servicios Residenciales,5837.0,11440.0
1,Servicios B2B Corporativos,10312.0,20211.0
2,Servicios de Valor Añadido (Cloud/Digital),3240.0,6350.0


In [19]:
# Agregar columnas de banda de confianza a la tabla de forecast ya calculada
margenes = dict(zip([f[0] for f in filas_ic], [f[2] for f in filas_ic]))
ingresos_forecast_2026['monto_min'] = ingresos_forecast_2026.apply(
    lambda r: round(r['monto_ingresos_forecast'] - margenes[r['concepto']], 2), axis=1)
ingresos_forecast_2026['monto_max'] = ingresos_forecast_2026.apply(
    lambda r: round(r['monto_ingresos_forecast'] + margenes[r['concepto']], 2), axis=1)
ingresos_forecast_2026.head(3)

,id_forecast,departamento_id,concepto,periodo_id,monto_ingresos_forecast,monto_min,monto_max
0,1,1,Servicios Residenciales,13,157972.93,146533.32,169412.54
1,2,1,Servicios Residenciales,14,160880.71,149441.10,172320.32
2,3,1,Servicios Residenciales,15,162400.90,150961.29,173840.51


In [20]:
# Guardar las 3 tablas de forecast para Power BI
ingresos_forecast_2026.to_csv('../data/processed/ingresos_forecast_2026.csv', index=False)
presupuesto_2026.to_csv('../data/processed/presupuesto_2026.csv', index=False)
gasto_esperado_2026.to_csv('../data/processed/gasto_esperado_2026.csv', index=False)

ingresos_forecast_2026.to_sql('ingresos_forecast_2026', conn, if_exists='replace', index=False)
presupuesto_2026.to_sql('presupuesto_2026', conn, if_exists='replace', index=False)
gasto_esperado_2026.to_sql('gasto_esperado_2026', conn, if_exists='replace', index=False)
conn.commit()
print('Guardado: ingresos_forecast_2026, presupuesto_2026, gasto_esperado_2026 (CSV + SQL)')

Guardado: ingresos_forecast_2026, presupuesto_2026, gasto_esperado_2026 (CSV + SQL)
